# Azure Machine Learning: controlled cloud lab

Follow [the full cloud lesson](../../docs/08-azure-lifecycle.md) and its setup lesson first.
Cloud actions allocate billable resources. **Do not use Run All for a live lab**:
run one stage, inspect its output, and fill in the resulting identifiers.

`RUN_CLOUD` defaults to `False`. The verification script also skips every cell
tagged `cloud`. Install the `azure` extra and select this project's kernel
before enabling cloud execution. Credentials come from your terminal login.


In [ ]:
import os
from iris_mlops.paths import PROJECT_DIR
import subprocess
import sys
from pathlib import Path

RUN_CLOUD = False

def step(module, *arguments):
    command = [sys.executable, "-m", "iris_mlops." + module, *map(str, arguments)]
    print(" ".join(command))
    if RUN_CLOUD:
        subprocess.run(command, check=True, cwd=PROJECT_DIR)
    else:
        print("Skipped: enable RUN_CLOUD only after completing setup and reviewing this step")

print("Cloud execution enabled:", RUN_CLOUD)


In [ ]:
IMAGE = os.getenv("AZURE_IMAGE", "replace-with-acr-uri@sha256:digest")
ENDPOINT = "replace-with-unique-endpoint-name"
JOB = "replace-after-training"
DOWNLOADED_RUN = "replace-with-directory-containing-run.json"
RUN_ID = "replace-after-import"
BUNDLE = Path("releases") / RUN_ID
VERSION = "1"
DEPLOYMENT = "blue"


## 1. Train and inspect the managed job

The local snapshot must already exist. Record the printed job name and inspect
its data/environment/outputs in Azure ML Studio. Update `JOB` above before download.


In [ ]:
step("azure_cloud", "train", "--image", IMAGE)


In [ ]:
step("azure_cloud", "download", JOB, "--output", "downloads/azure")


## 2. Import, evaluate and review

Set `DOWNLOADED_RUN` to the printed directory. Import prints the run ID; update
`RUN_ID` and `BUNDLE` before evaluating. Inspect the gate before promoting.


In [ ]:
step("workflow", "import", DOWNLOADED_RUN)


In [ ]:
step("workflow", "evaluate", RUN_ID)


In [ ]:
step("workflow", "promote", RUN_ID)
step("workflow", "export", RUN_ID, "--output", BUNDLE)
step("azure_cloud", "register", "--bundle", BUNDLE, "--version", VERSION, "--image", IMAGE)


## 3. Stage, inspect, then switch

Create the endpoint once. Use a new deployment/version for later releases.
The staging operation does not change traffic.


In [ ]:
step("azure_cloud", "create-endpoint", "--endpoint", ENDPOINT)
step("azure_cloud", "deploy", "--endpoint", ENDPOINT, "--deployment", DEPLOYMENT, "--version", VERSION)
step("azure_cloud", "smoke", "--endpoint", ENDPOINT, "--deployment", DEPLOYMENT)


In [ ]:
step("azure_cloud", "switch", "--endpoint", ENDPOINT, "--deployment", DEPLOYMENT)
step("azure_cloud", "status", "--endpoint", ENDPOINT)
step("azure_cloud", "smoke", "--endpoint", ENDPOINT)


## 4. Monitor and practice recovery

Enable diagnostic settings and run the KQL queries in the lesson. After a green
release, switch back to the retained blue deployment to demonstrate rollback.

Use the explicit v2 steps in [lesson 08, step 4](../../docs/08-azure-lifecycle.md#4-ship-a-second-version-and-roll-back).
Record the v1 run ID before changing configuration. After cloud rollback,
restore that run with `uv run iris rollback RUN_ID_V1` in the terminal.


In [ ]:
step("azure_cloud", "logs", "--endpoint", ENDPOINT, "--deployment", DEPLOYMENT)
# After staging/promoting green, run the next call to restore blue:
# step("azure_cloud", "switch", "--endpoint", ENDPOINT, "--deployment", "blue")


## 5. Cleanup

Delete the endpoint when finished. Also delete the dedicated resource group
using the lesson's cleanup command; the following cell only removes the endpoint.


In [ ]:
# Uncomment when the endpoint is no longer needed:
# step("azure_cloud", "delete-endpoint", "--endpoint", ENDPOINT)
